## Setup & Data Loading

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

ROOT = Path("/home/agent06/Desktop/aivis-group-noise")
if not ROOT.exists():
    ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.esc import CLASSES, GROUP_OF

RESULTS_DIR = ROOT / "results"

# Names and colors for the 5 broad sound families in ESC-50
GROUP_LABELS = {
    "animals": "Animals",
    "natural": "Natural Sounds",
    "human": "Human Sounds",
    "domestic": "Household / Domestic",
    "urban": "Urban / Street",
}
GROUP_ORDER = ["animals", "natural", "human", "domestic", "urban"]
CLASS_RANK = {c: i for i, c in enumerate(CLASSES)}

OUTCOME_COLORS = {
    "Correct": "#10b981",                 # Green
    "urban": "#ef4444",                   # Red
    "animals (wrong class)": "#f59e0b",   # Orange
    "natural": "#3b82f6",                 # Blue
    "human": "#8b5cf6",                   # Purple
    "domestic": "#8c564b",                # Brown
}

MACRO_COLORS = {
    "animals": "#10b981",
    "natural": "#3b82f6",
    "human": "#8b5cf6",
    "domestic": "#8c564b",
    "urban": "#ef4444",
}

def plotly_export_config(filename: str) -> dict:
    """Configures Plotly's camera download button to save with a descriptive filename instead of 'newplot'."""
    clean_name = re.sub(r"[^a-zA-Z0-9_.-]+", "_", filename).strip("_").lower()
    return {
        "toImageButtonOptions": {
            "format": "png",
            "filename": clean_name,
            "scale": 2,
        }
    }

# Load prediction tables
df_perturb = pd.read_csv(RESULTS_DIR / "perturbation_predictions.csv")
df_base = pd.read_csv(RESULTS_DIR / "baseline_predictions.csv")
df_imit = pd.read_csv(RESULTS_DIR / "imitations_predictions.csv")
df_lyre = pd.read_csv(RESULTS_DIR / "lyrebird_predictions.csv")

# Standardize columns across datasets
df_perturb["group"] = df_perturb["category"].map(GROUP_OF).fillna("animals")

def _infer_imitation_category(fname):
    f = str(fname).lower()
    for key, target_cat in [("bee", "insects"), ("chicken", "hen"), ("bird", "chirping_birds"),
                            ("pig", "pig"), ("dog", "dog"), ("sheep", "sheep")]:
        if key in f:
            return target_cat
    return "unknown"

df_imit["filename"] = df_imit["file"] + " (t=" + df_imit["t_start"].astype(str) + "s)"
df_imit["category"] = df_imit["file"].map(_infer_imitation_category)
df_imit["group"] = df_imit["category"].map(GROUP_OF).fillna("animals")
df_imit["experiment"] = "human_imitation"
df_imit["param"] = df_imit["t_start"]
df_imit["distractor"] = np.nan
df_imit["correct"] = df_imit["pred"] == df_imit["category"]

df_lyre["filename"] = df_lyre["file"].str.slice(0, 28) + "... (t=" + df_lyre["t_start"].astype(str) + "s)"
df_lyre["category"] = "chirping_birds"
df_lyre["group"] = "natural"
df_lyre["experiment"] = "lyrebird_recording"
df_lyre["param"] = df_lyre["t_start"]
df_lyre["distractor"] = np.nan
df_lyre["correct"] = df_lyre["pred"].isin(["chirping_birds", "crow"])

df_base["experiment"] = "clean_baseline"
df_base["param"] = 0.0
df_base["distractor"] = np.nan

DATASETS = {
    "Perturbed Animal Clips (1,480 clips)": df_perturb,
    "Clean Baseline — Animals Only (400 clips)": df_base[df_base["group"] == "animals"].copy(),
    "Clean Baseline — All 50 Classes (2,000 clips)": df_base,
    "Human Vocal Imitations (20 clips)": df_imit,
    "Superb Lyrebird Mimicry (159 clips)": df_lyre,
}

DATASET_SLUGS = {
    "Perturbed Animal Clips (1,480 clips)": "perturbed_animals",
    "Clean Baseline — Animals Only (400 clips)": "clean_baseline_animals",
    "Clean Baseline — All 50 Classes (2,000 clips)": "clean_baseline_all_esc50",
    "Human Vocal Imitations (20 clips)": "human_vocal_imitations",
    "Superb Lyrebird Mimicry (159 clips)": "superb_lyrebird_mimicry",
}

rng = np.random.default_rng(42)
for d in DATASETS.values():
    d["_jitter"] = rng.uniform(-0.26, 0.26, size=len(d))

print(f"Loaded {len(DATASETS)} datasets successfully from {RESULTS_DIR.name}/.")

Loaded 5 datasets successfully from results/.


## 1. Cumulative Confidence Sweep: Tracking Prediction Drift

This line chart tracks what happens across confidence thresholds (`0.1` to `1.0`) for **Clean**, **White Noise**, and **Mixed Audio**:
* **X-axis:** Animal Confidence threshold (`0.1` to `1.0`).
* **Y-axis:** Percentage of animal clips predicted as **Urban**, **Human**, or **Animal**.
* As animal confidence drops, the **Predicted as Urban** line spikes sharply upward under **Mixed Audio** and **White Noise**, while **Clean** recordings stay at `0%`.

In [2]:
thresholds = np.round(np.arange(0.10, 1.001, 0.05), 2)

EXP_STYLES = {
    "clean": {"label": "Clean (Baseline)", "color": "#10b981", "dash": "solid", "symbol": "circle"},
    "noise": {"label": "White Noise", "color": "#f59e0b", "dash": "dash", "symbol": "triangle-up"},
    "mix":   {"label": "Mixed Audio (Competing Sound)", "color": "#ef4444", "dash": "solid", "symbol": "square"},
}

sweep_rows = []
for exp in ["clean", "noise", "mix"]:
    sub_exp = df_perturb[df_perturb["experiment"] == exp]
    for t in thresholds:
        s = sub_exp[sub_exp["p_animal_max"] <= t]
        n = len(s)
        sweep_rows.append({
            "experiment": exp,
            "threshold": t,
            "n": n,
            "urban": (s["pred_group"] == "urban").mean() * 100.0 if n > 0 else np.nan,
            "human": (s["pred_group"] == "human").mean() * 100.0 if n > 0 else np.nan,
            "animals": (s["pred_group"] == "animals").mean() * 100.0 if n > 0 else np.nan,
        })
curves_df = pd.DataFrame(sweep_rows)

w_sweep_view = widgets.Dropdown(
    options=[
        ("All 3 Panels Side-by-Side (Urban, Human, Animal)", "all3"),
        ("Single Chart: Focus on One Category", "single"),
    ],
    value="all3",
    description="View:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="420px"),
)

w_target_grp = widgets.Dropdown(
    options=[
        ("Predicted as Urban (%)", "urban"),
        ("Predicted as Human (%)", "human"),
        ("Predicted as Animal (%)", "animals"),
    ],
    value="urban",
    description="Category:",
    style={"description_width": "75px"},
    layout=widgets.Layout(width="250px"),
)

w_reverse_x = widgets.Checkbox(
    value=False,
    description="Reverse X-axis (1.0 → 0.1 left-to-right to read as 'Confidence Dropping')",
    indent=False,
    layout=widgets.Layout(width="470px"),
)

out_sweep = widgets.Output()

def render_sweep_chart(view_mode, target_grp, reverse_x):
    with out_sweep:
        out_sweep.clear_output(wait=True)
        w_target_grp.disabled = (view_mode == "all3")

        if view_mode == "all3":
            fig_confidence_sweep = make_subplots(
                rows=1, cols=3,
                subplot_titles=(
                    "% Predicted as URBAN",
                    "% Predicted as HUMAN",
                    "% Predicted as ANIMAL",
                ),
                horizontal_spacing=0.07,
            )
            for grp_col, col_idx in [("urban", 1), ("human", 2), ("animals", 3)]:
                for exp, st in EXP_STYLES.items():
                    c = curves_df[curves_df["experiment"] == exp]
                    fig_confidence_sweep.add_trace(
                        go.Scatter(
                            x=c["threshold"],
                            y=c[grp_col],
                            mode="lines+markers",
                            name=st["label"],
                            showlegend=(col_idx == 1),
                            line=dict(color=st["color"], width=3, dash=st["dash"]),
                            marker=dict(size=7, symbol=st["symbol"]),
                            customdata=c[["n"]],
                            hovertemplate=(
                                f"<b>{st['label']}</b><br>"
                                "Confidence ≤ %{x:.2f}<br>"
                                f"Predicted {grp_col.capitalize()}: <b>%{{y:.1f}}%</b> (n=%{{customdata[0]}})<extra></extra>"
                            ),
                        ),
                        row=1, col=col_idx,
                    )
                fig_confidence_sweep.update_xaxes(
                    title_text="Confidence Threshold (≤ τ)",
                    autorange="reversed" if reverse_x else True,
                    dtick=0.2,
                    row=1, col=col_idx,
                )
            fig_confidence_sweep.update_yaxes(title_text="% of Animal Clips", ticksuffix="%", row=1, col=1)
            fig_confidence_sweep.update_yaxes(ticksuffix="%", row=1, col=2)
            fig_confidence_sweep.update_yaxes(ticksuffix="%", range=[-2, 105], row=1, col=3)
            sweep_filename = "cumulative_confidence_sweep_urban_human_animal"
        else:
            title_map = {
                "urban": "% of Animal Clips Predicted as URBAN",
                "human": "% of Animal Clips Predicted as HUMAN",
                "animals": "% of Animal Clips Predicted as ANIMAL",
            }
            fig_confidence_sweep = go.Figure()
            for exp, st in EXP_STYLES.items():
                c = curves_df[curves_df["experiment"] == exp]
                fig_confidence_sweep.add_trace(
                    go.Scatter(
                        x=c["threshold"],
                        y=c[target_grp],
                        mode="lines+markers",
                        name=st["label"],
                        line=dict(color=st["color"], width=3.5, dash=st["dash"]),
                        marker=dict(size=8, symbol=st["symbol"]),
                        customdata=c[["n"]],
                        hovertemplate=(
                            f"<b>{st['label']}</b><br>"
                            "Confidence ≤ %{x:.2f}<br>"
                            f"{title_map[target_grp]}: <b>%{{y:.1f}}%</b> (n=%{{customdata[0]}})<extra></extra>"
                        ),
                    )
                )
            fig_confidence_sweep.update_xaxes(
                title_text="Confidence Threshold (≤ τ)",
                autorange="reversed" if reverse_x else True,
                dtick=0.1,
            )
            fig_confidence_sweep.update_yaxes(title_text=title_map[target_grp], ticksuffix="%")
            sweep_filename = f"cumulative_confidence_sweep_predicted_{target_grp}"

        fig_confidence_sweep.update_layout(
            height=450,
            width=1220,
            template="plotly_white",
            title=dict(
                text="<b>Cumulative Confidence Sweep:</b> As Animal Confidence Drops, Predictions Drift into Urban & Human Categories",
                font=dict(size=15),
            ),
            legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="left", x=0),
            margin=dict(l=60, r=40, t=105, b=65),
        )
        fig_confidence_sweep.show(config=plotly_export_config(sweep_filename))

sweep_controls = widgets.HBox([w_sweep_view, w_target_grp, w_reverse_x])
widgets.interactive_output(
    render_sweep_chart,
    {"view_mode": w_sweep_view, "target_grp": w_target_grp, "reverse_x": w_reverse_x},
)
display(widgets.VBox([sweep_controls, out_sweep]))

## 2. Cross-Dataset Comparison & Complete Breakdown Table

Below is a side-by-side comparison across all five test sets (**Clean Animals**, **White Noise**, **Mixed Audio**, **Human Imitations**, and **Superb Lyrebird**), followed by a complete summary table showing accuracy and mistake types for every experimental condition and background sound.

In [3]:
comp_records = []
for label, subset in [
    ("1. Clean Animals", df_base[df_base["group"] == "animals"]),
    ("2. White Noise", df_perturb[df_perturb["experiment"] == "noise"]),
    ("3. Mixed Audio", df_perturb[df_perturb["experiment"] == "mix"]),
    ("4. Human Imitations", df_imit),
    ("5. Superb Lyrebird", df_lyre),
]:
    for _, r in subset.iterrows():
        comp_records.append({
            "Dataset": label,
            "p_animal_max": r["p_animal_max"],
            "pred_group": r["pred_group"],
        })
df_comp = pd.DataFrame(comp_records)

fig_cross_dataset_comparison = make_subplots(
    rows=1, cols=2,
    column_widths=[0.48, 0.52],
    subplot_titles=(
        "Animal Confidence Score Distribution",
        "Share of Predicted Sound Families (% of Clips)",
    ),
    horizontal_spacing=0.12,
)

for ds_label in df_comp["Dataset"].unique():
    s = df_comp[df_comp["Dataset"] == ds_label]
    fig_cross_dataset_comparison.add_trace(
        go.Box(
            y=s["p_animal_max"],
            name=ds_label,
            boxmean=True,
            boxpoints="outliers",
            showlegend=False,
        ),
        row=1, col=1,
    )

group_pct = df_comp.groupby(["Dataset", "pred_group"]).size().unstack(fill_value=0)
group_pct = group_pct.div(group_pct.sum(axis=1), axis=0) * 100.0

for grp_name, col in MACRO_COLORS.items():
    if grp_name in group_pct.columns:
        fig_cross_dataset_comparison.add_trace(
            go.Bar(
                x=group_pct.index.tolist(),
                y=group_pct[grp_name].values,
                name=GROUP_LABELS.get(grp_name, grp_name),
                marker_color=col,
                text=[f"{v:.1f}%" if v >= 5 else "" for v in group_pct[grp_name].values],
                textposition="inside",
            ),
            row=1, col=2,
        )

fig_cross_dataset_comparison.update_layout(
    barmode="stack",
    height=440,
    width=1200,
    template="plotly_white",
    title="<b>Cross-Dataset Comparison:</b> Clean Animals vs. Perturbed Audio vs. Imitations & Lyrebird",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="right", x=1.0),
    margin=dict(l=60, r=40, t=100, b=90),
)
fig_cross_dataset_comparison.update_xaxes(tickangle=-20, row=1, col=1)
fig_cross_dataset_comparison.update_yaxes(title_text="Animal Confidence (p_animal_max)", row=1, col=1)
fig_cross_dataset_comparison.update_xaxes(tickangle=-20, row=1, col=2)
fig_cross_dataset_comparison.update_yaxes(title_text="Share of Predictions (%)", range=[0, 100], row=1, col=2)
fig_cross_dataset_comparison.show(
    config=plotly_export_config("cross_dataset_comparison_confidence_and_sound_families")
)

# Polished Summary Table by Condition & Background Sound
exp_breakdown = (
    df_perturb.assign(
        distractor=lambda d: d["distractor"].fillna("—"),
        is_error=lambda d: ~d["correct"].astype(bool),
        in_group_err=lambda d: (~d["correct"].astype(bool)) & (d["group"] == d["pred_group"]),
        cross_group_err=lambda d: (~d["correct"].astype(bool)) & (d["group"] != d["pred_group"]),
    )
    .groupby(["experiment", "distractor"], as_index=False)
    .agg(
        Clips=("filename", "count"),
        Accuracy_Pct=("correct", lambda s: round(s.mean() * 100.0, 1)),
        Total_Mistakes=("is_error", "sum"),
        Same_Group_Mistakes=("in_group_err", "sum"),
        Cross_Group_Mistakes=("cross_group_err", "sum"),
        Avg_Animal_Conf=("p_animal_max", lambda s: round(s.mean(), 3)),
        Avg_Top1_Conf=("conf", lambda s: round(s.mean(), 3)),
    )
    .rename(columns={
        "experiment": "Condition",
        "distractor": "Background Sound",
        "Accuracy_Pct": "Accuracy (%)",
        "Total_Mistakes": "Total Mistakes",
        "Same_Group_Mistakes": "Same-Group Mistakes",
        "Cross_Group_Mistakes": "Cross-Group Mistakes",
        "Avg_Animal_Conf": "Avg Animal Confidence",
        "Avg_Top1_Conf": "Avg Top-1 Confidence",
    })
)

display(HTML("<h4>Summary Table: Accuracy & Mistake Types by Condition and Background Sound</h4>"))
display(exp_breakdown.style.hide(axis="index").format({
    "Accuracy (%)": "{:.1f}%",
    "Avg Animal Confidence": "{:.3f}",
    "Avg Top-1 Confidence": "{:.3f}",
}))

Condition,Background Sound,Clips,Accuracy (%),Total Mistakes,Same-Group Mistakes,Cross-Group Mistakes,Avg Animal Confidence,Avg Top-1 Confidence
clean,—,40,100.0%,0,0,0,0.772,0.714
gain,—,200,100.0%,0,0,0,0.773,0.722
mix,cat,20,40.0%,12,11,1,0.688,0.408
mix,chirping_birds,20,100.0%,0,0,0,0.695,0.695
mix,cow,40,27.5%,29,29,0,0.815,0.693
mix,crickets,20,15.0%,17,0,17,0.963,0.824
mix,crying_baby,200,39.5%,121,0,121,0.290,0.611
mix,dog,20,30.0%,14,14,0,0.614,0.358
mix,engine,200,91.0%,18,3,15,0.578,0.540
mix,hen,20,55.0%,9,9,0,0.932,0.765


## 3. Interactive Confusion Matrix & Mistake Breakdown

Use the controls below to filter audio clips by **dataset**, **animal confidence threshold**, **perturbation type** (`clean`, `gain`, `noise`, `mix`), and **competing background sound**.
* **Left chart:** Each dot is a single audio clip, colored by the sound family the model predicted.
* **Right chart:** The **Confusion Matrix**, organized by sound family (*Animals, Natural, Human, Household, Urban*) with solid divider lines showing where mistakes cross category boundaries.
* **Bottom summary card:** Shows the exact percentage of **Same-Group Mistakes** vs. **Cross-Group Mistakes** for your current selection.

In [4]:
def order_matrix_by_sound_family(cm_df):
    """Orders confusion matrix rows and columns by ESC-50 sound family."""
    sort_key = lambda c: (
        GROUP_ORDER.index(GROUP_OF.get(c, "animals")),
        CLASS_RANK.get(c, 999),
        str(c),
    )
    row_order = sorted(cm_df.index.tolist(), key=sort_key)
    col_order = sorted(cm_df.columns.tolist(), key=sort_key)
    return cm_df.reindex(index=row_order, columns=col_order, fill_value=0)

# UI Controls
w_dataset = widgets.Dropdown(
    options=list(DATASETS.keys()),
    value="Perturbed Animal Clips (1,480 clips)",
    description="Dataset:",
    style={"description_width": "65px"},
    layout=widgets.Layout(width="360px"),
)

w_threshold = widgets.FloatSlider(
    value=0.72,
    min=0.0,
    max=1.0,
    step=0.02,
    description="Confidence:",
    continuous_update=False,
    style={"description_width": "85px"},
    layout=widgets.Layout(width="300px"),
)

w_mode = widgets.Dropdown(
    options=[("≤ threshold", "le"), ("≥ threshold", "ge")],
    value="le",
    description="Filter:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="175px"),
)

w_only_errors = widgets.Checkbox(
    value=True,
    description="Show mistakes only",
    indent=False,
    layout=widgets.Layout(width="180px"),
)

w_experiment = widgets.Dropdown(
    options=["All", "clean", "noise", "gain", "mix"],
    value="All",
    description="Condition:",
    style={"description_width": "75px"},
    layout=widgets.Layout(width="190px"),
)

w_param = widgets.Dropdown(
    options=["All"],
    value="All",
    description="Level (dB):",
    style={"description_width": "75px"},
    layout=widgets.Layout(width="180px"),
)

w_distractor = widgets.Dropdown(
    options=["All"],
    value="All",
    description="Background Sound:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="250px"),
)

def _sync_filters(*args):
    ds_name = w_dataset.value
    d = DATASETS[ds_name]
    is_perturb = ds_name.startswith("Perturbed")
    w_experiment.disabled = not is_perturb
    w_param.disabled = not is_perturb
    w_distractor.disabled = not is_perturb
    if is_perturb:
        exps = ["All"] + sorted(d["experiment"].dropna().astype(str).unique().tolist())
        if list(w_experiment.options) != exps:
            w_experiment.options = exps
        exp = w_experiment.value
        sub = d if exp == "All" else d[d["experiment"].astype(str) == exp]
        params = ["All"] + [str(p) for p in sorted(sub["param"].dropna().unique(), reverse=True)]
        w_param.options = params
        if w_param.value not in params:
            w_param.value = "All"
        dists = ["All"] + sorted(d["distractor"].dropna().astype(str).unique().tolist())
        w_distractor.options = dists
        w_distractor.disabled = exp not in ("All", "mix")
        if w_distractor.disabled:
            w_distractor.value = "All"
    else:
        w_experiment.value = "All"
        w_param.value = "All"
        w_distractor.value = "All"

w_dataset.observe(_sync_filters, names="value")
w_experiment.observe(_sync_filters, names="value")
_sync_filters()

out_cm = widgets.Output()

def render_confusion_matrix(dataset_name, threshold, filter_mode, experiment, param, distractor, only_errors):
    with out_cm:
        out_cm.clear_output(wait=True)
        sub_all = DATASETS[dataset_name].copy()

        if dataset_name.startswith("Perturbed"):
            if experiment != "All":
                sub_all = sub_all[sub_all["experiment"].astype(str) == experiment]
            if param != "All":
                sub_all = sub_all[sub_all["param"].astype(str) == param]
            if distractor != "All":
                sub_all = sub_all[sub_all["distractor"].astype(str) == distractor]

        if filter_mode == "le":
            slice_all = sub_all[sub_all["p_animal_max"] <= threshold].copy()
            cond_str = f"Animal Confidence ≤ {threshold:.2f}"
        else:
            slice_all = sub_all[sub_all["p_animal_max"] >= threshold].copy()
            cond_str = f"Animal Confidence ≥ {threshold:.2f}"

        filtered = slice_all[~slice_all["correct"].astype(bool)].copy() if only_errors else slice_all.copy()

        if filtered.empty:
            print(
                f"No clips match the current filter ({cond_str}). "
                f"Try unchecking 'Show mistakes only' or raising the confidence slider."
            )
            return

        def classify_outcome(r):
            if bool(r["correct"]):
                return "Correct"
            if r["pred_group"] == "animals":
                return "animals (wrong class)"
            return str(r["pred_group"])

        filtered["outcome"] = filtered.apply(classify_outcome, axis=1)

        max_cols = 25 if "All 50" in dataset_name else 18
        top_preds = filtered["pred"].value_counts().head(max_cols).index.tolist()
        base_categories = sorted(filtered["category"].dropna().unique().tolist())

        cm_raw = (
            filtered[filtered["pred"].isin(top_preds)]
            .groupby(["category", "pred"])
            .size()
            .unstack(fill_value=0)
            .reindex(index=base_categories, columns=top_preds, fill_value=0)
        )
        cm_df = order_matrix_by_sound_family(cm_raw)
        ordered_cats = cm_df.index.tolist()
        cat_to_idx = {c: i for i, c in enumerate(ordered_cats)}
        filtered["x_pos"] = filtered["category"].map(cat_to_idx) + filtered["_jitter"]

        fig_confusion_matrix = make_subplots(
            rows=1, cols=2,
            column_widths=[0.45, 0.55],
            subplot_titles=(
                f"Individual Audio Clips ({len(filtered)} shown)",
                "Confusion Matrix (Grouped by Sound Family)",
            ),
            horizontal_spacing=0.13,
        )

        # Left Panel: Clip-level scatter plot
        for outcome_label, color in OUTCOME_COLORS.items():
            grp = filtered[filtered["outcome"] == outcome_label]
            if grp.empty:
                continue
            hover_text = [
                f"<b>File:</b> {r['filename']}<br>"
                f"<b>True Animal:</b> {r['category']}<br>"
                f"<b>Model Predicted:</b> {r['pred']} ({GROUP_LABELS.get(r['pred_group'], r['pred_group'])})<br>"
                f"<b>Animal Confidence:</b> {r['p_animal_max']:.3f} | <b>Top-1 Score:</b> {r['conf']:.3f}<br>"
                f"<b>Condition:</b> {r['experiment']} ({r['param']} dB)"
                + (f" + {r['distractor']}" if pd.notna(r.get('distractor', np.nan)) else "")
                for _, r in grp.iterrows()
            ]
            fig_confusion_matrix.add_trace(
                go.Scatter(
                    x=grp["x_pos"],
                    y=grp["p_animal_max"],
                    mode="markers",
                    name=f"{outcome_label} ({len(grp)})",
                    marker=dict(size=8, color=color, opacity=0.82, line=dict(width=0.5, color="white")),
                    hovertemplate="%{text}<extra></extra>",
                    text=hover_text,
                ),
                row=1, col=1,
            )

        fig_confusion_matrix.add_hline(
            y=threshold, line_dash="dash", line_color="#0f172a",
            annotation_text=f"Threshold ({threshold:.2f})",
            annotation_position="top left",
            row=1, col=1,
        )

        # Right Panel: Confusion Matrix Heatmap
        pred_to_group = {c: GROUP_OF.get(c, "animals") for c in cm_df.columns}
        x_labels = [
            f"<b>{p} [urban]</b>" if pred_to_group.get(p) == "urban" else f"{p} [{pred_to_group.get(p, '')}]"
            for p in cm_df.columns
        ]
        y_labels = [
            f"{c} [{GROUP_OF.get(c, 'animals')}]"
            for c in cm_df.index
        ]

        fig_confusion_matrix.add_trace(
            go.Heatmap(
                z=cm_df.values,
                x=x_labels,
                y=y_labels,
                colorscale="YlOrRd",
                text=cm_df.values,
                texttemplate="%{text}",
                hovertemplate="True Sound: %{y}<br>Predicted: %{x}<br>Clips: %{z}<extra></extra>",
                colorbar=dict(title="Clips", len=0.85),
            ),
            row=1, col=2,
        )

        # Sound family block headers and boundary lines
        col_groups = [pred_to_group.get(c, "animals") for c in cm_df.columns]
        row_groups = [GROUP_OF.get(r, "animals") for r in cm_df.index]

        start_c = 0
        for i in range(1, len(col_groups) + 1):
            if i == len(col_groups) or col_groups[i] != col_groups[start_c]:
                grp_key = col_groups[start_c]
                mid_x = (start_c + i - 1) / 2.0
                fig_confusion_matrix.add_annotation(
                    x=mid_x, y=-0.75, xref="x2", yref="y2",
                    text=f"<b>{GROUP_LABELS.get(grp_key, grp_key)}</b>",
                    showarrow=False,
                    font=dict(size=10, color="#0f172a"),
                    bgcolor="rgba(241, 245, 249, 0.9)",
                    bordercolor="#475569",
                    borderwidth=1,
                )
                if i < len(col_groups):
                    fig_confusion_matrix.add_vline(x=i - 0.5, line_width=2.2, line_color="#1e293b", row=1, col=2)
                start_c = i

        for j in range(1, len(row_groups)):
            if row_groups[j] != row_groups[j - 1]:
                fig_confusion_matrix.add_hline(y=j - 0.5, line_width=2.2, line_color="#1e293b", row=1, col=2)

        fig_confusion_matrix.update_xaxes(
            tickmode="array",
            tickvals=list(range(len(ordered_cats))),
            ticktext=ordered_cats,
            title_text="True Animal / Sound Class",
            tickangle=-35,
            row=1, col=1,
        )
        fig_confusion_matrix.update_yaxes(
            title_text="Animal Confidence (p_animal_max)",
            range=[-0.03, max(1.03, threshold + 0.05)],
            row=1, col=1,
        )
        fig_confusion_matrix.update_xaxes(title_text="Model Prediction [Sound Family]", tickangle=-40, row=1, col=2)
        fig_confusion_matrix.update_yaxes(title_text="True Sound [Sound Family]", autorange="reversed", row=1, col=2)

        urban_rate = (filtered["pred_group"] == "urban").mean()
        human_rate = (filtered["pred_group"] == "human").mean()
        mode_badge = "MISTAKES ONLY" if only_errors else "ALL CLIPS"
        fig_confusion_matrix.update_layout(
            height=600,
            width=1240,
            template="plotly_white",
            title=dict(
                text=(
                    f"<b>{dataset_name}</b> [{mode_badge}] | {cond_str} | "
                    f"Predicted Urban: <b style='color:#ef4444'>{urban_rate:.1%}</b> | "
                    f"Predicted Human: <b style='color:#8b5cf6'>{human_rate:.1%}</b>"
                ),
                font=dict(size=14),
            ),
            legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="left", x=0),
            margin=dict(l=60, r=40, t=120, b=115),
        )

        # Build descriptive download filename for the camera button
        ds_slug = DATASET_SLUGS.get(dataset_name, "dataset")
        exp_part = f"_{experiment}" if (dataset_name.startswith("Perturbed") and experiment != "All") else ""
        dist_part = f"_{distractor}" if (dataset_name.startswith("Perturbed") and distractor != "All") else ""
        err_part = "_mistakes_only" if only_errors else "_all_clips"
        cm_filename = f"confusion_matrix_{ds_slug}{exp_part}{dist_part}_conf_{filter_mode}_{threshold:.2f}{err_part}"

        fig_confusion_matrix.show(config=plotly_export_config(cm_filename))

        # Summary Card: Same-Group vs. Cross-Group Mistakes
        errors_df = slice_all[~slice_all["correct"].astype(bool)].copy()
        n_total_slice = len(slice_all)
        n_err = len(errors_df)
        if n_err > 0:
            in_group_mask = errors_df["group"].astype(str) == errors_df["pred_group"].astype(str)
            n_in_group = int(in_group_mask.sum())
            n_cross_group = int((~in_group_mask).sum())
            in_group_pct = 100.0 * n_in_group / n_err
            cross_group_pct = 100.0 * n_cross_group / n_err

            cross_targets = errors_df[~in_group_mask]["pred_group"].value_counts().to_dict()
            cross_breakdown_str = ", ".join(
                f"<b>{GROUP_LABELS.get(k, k)}:</b> {v} ({100.0*v/n_err:.1f}%)"
                for k, v in cross_targets.items()
            ) or "None"
        else:
            n_in_group = n_cross_group = 0
            in_group_pct = cross_group_pct = 0.0
            cross_breakdown_str = "No mistakes in current selection"

        card_html = f"""
        <div style="border: 1px solid #cbd5e1; border-left: 5px solid #2563eb; border-radius: 6px;
                    padding: 12px 16px; background: #f8fafc; font-family: sans-serif; width: 1180px; margin-top: 6px;">
            <div style="font-size: 14px; font-weight: 700; color: #0f172a; margin-bottom: 6px;">
                Mistake Breakdown in Current Selection ({n_err} mistakes out of {n_total_slice} clips)
            </div>
            <div style="display: flex; gap: 32px; font-size: 13px; color: #1e293b;">
                <div>
                    <span style="color: #d97706; font-weight: 700;">● Same-Group Mistakes (e.g. hen → rooster):</span>
                    <b>{in_group_pct:.1f}%</b> ({n_in_group} / {n_err})
                </div>
                <div>
                    <span style="color: #dc2626; font-weight: 700;">● Cross-Group Mistakes (e.g. dog → siren [urban]):</span>
                    <b>{cross_group_pct:.1f}%</b> ({n_cross_group} / {n_err})
                </div>
            </div>
            <div style="font-size: 12px; color: #475569; margin-top: 6px;">
                Where Cross-Group Mistakes Go: {cross_breakdown_str}
            </div>
        </div>
        """
        display(HTML(card_html))

row1 = widgets.HBox([w_dataset, w_threshold, w_mode, w_only_errors])
row2 = widgets.HBox([w_experiment, w_param, w_distractor])
widgets.interactive_output(
    render_confusion_matrix,
    {
        "dataset_name": w_dataset,
        "threshold": w_threshold,
        "filter_mode": w_mode,
        "experiment": w_experiment,
        "param": w_param,
        "distractor": w_distractor,
        "only_errors": w_only_errors,
    },
)
display(widgets.VBox([row1, row2, out_cm]))